In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("./CSV/house_prices.csv")

print(df.shape)
print(df.columns)
df.head()

(187531, 21)
Index(['Index', 'Title', 'Description', 'Amount(in rupees)',
       'Price (in rupees)', 'location', 'Carpet Area', 'Status', 'Floor',
       'Transaction', 'Furnishing', 'facing', 'overlooking', 'Society',
       'Bathroom', 'Balcony', 'Car Parking', 'Ownership', 'Super Area',
       'Dimensions', 'Plot Area'],
      dtype='str')


,Index,Title,Description,Amount(in rupees),Price (in rupees),location,Carpet Area,Status,Floor,Transaction,...,facing,overlooking,Society,Bathroom,Balcony,Car Parking,Ownership,Super Area,Dimensions,Plot Area
0,0,1 BHK Ready to Occupy Flat for sale in Srushti...,"Bhiwandi, Thane has an attractive 1 BHK Flat f...",42 Lac,6000.0,thane,500 sqft,Ready to Move,10 out of 11,Resale,...,NaN,NaN,Srushti Siddhi Mangal Murti Complex,1,2,NaN,NaN,NaN,NaN,NaN
1,1,2 BHK Ready to Occupy Flat for sale in Dosti V...,One can find this stunning 2 BHK flat for sale...,98 Lac,13799.0,thane,473 sqft,Ready to Move,3 out of 22,Resale,...,East,Garden/Park,Dosti Vihar,2,NaN,1 Open,Freehold,NaN,NaN,NaN
2,2,2 BHK Ready to Occupy Flat for sale in Sunrise...,Up for immediate sale is a 2 BHK apartment in ...,1.40 Cr,17500.0,thane,779 sqft,Ready to Move,10 out of 29,Resale,...,East,Garden/Park,Sunrise by Kalpataru,2,NaN,1 Covered,Freehold,NaN,NaN,NaN
3,3,1 BHK Ready to Occupy Flat for sale Kasheli,This beautiful 1 BHK Flat is available for sal...,25 Lac,NaN,thane,530 sqft,Ready to Move,1 out of 3,Resale,...,NaN,NaN,NaN,1,1,NaN,NaN,NaN,NaN,NaN
4,4,2 BHK Ready to Occupy Flat for sale in TenX Ha...,"This lovely 2 BHK Flat in Pokhran Road, Thane ...",1.60 Cr,18824.0,thane,635 sqft,Ready to Move,20 out of 42,Resale,...,West,"Garden/Park, Main Road",TenX Habitat Raymond Realty,2,NaN,1 Covered,Co-operative Society,NaN,NaN,NaN


In [2]:
df['Amount(in rupees)'].head(20)

0       42 Lac 
1       98 Lac 
2      1.40 Cr 
3       25 Lac 
4      1.60 Cr 
5       45 Lac 
6     16.5 Lac 
7       60 Lac 
8       60 Lac 
9      1.60 Cr 
10     1.40 Cr 
11     1.36 Cr 
12     1.35 Cr 
13     4.25 Cr 
14      75 Lac 
15      90 Lac 
16      37 Lac 
17      35 Lac 
18      90 Lac 
19      35 Lac 
Name: Amount(in rupees), dtype: str

In [3]:
def convert_price(x):
    x = str(x).replace('₹', '').strip()
    
    if 'Cr' in x:
        return float(x.replace('Cr', '').strip()) * 1e7
    elif 'Lac' in x:
        return float(x.replace('Lac', '').strip()) * 1e5
    else:
        return np.nan

In [4]:
df['price'] = df['Amount(in rupees)'].apply(convert_price)

In [5]:
print("Valid prices:", df['price'].notnull().sum())
print("Total rows:", len(df))

Valid prices: 177847
Total rows: 187531


In [6]:
df = df[df['price'].notnull()]
print(df.shape)

(177847, 22)


In [7]:
df['price'] = np.log(df['price'])

In [8]:
print(df.shape)
print(df['price'].head())
print(df['price'].describe())

(177847, 22)
0    15.250595
1    16.097893
2    16.454568
3    14.731801
4    16.588099
Name: price, dtype: float64
count    177847.000000
mean         15.923140
std           0.821877
min          11.512925
25%          15.392425
50%          15.869634
75%          16.489659
max          23.362537
Name: price, dtype: float64


In [9]:
for col in ['Furnishing', 'Transaction', 'Status']:
    print("\n", col)
    print(df[col].value_counts(dropna=False).head(10))


 Furnishing
Furnishing
Semi-Furnished    82868
Unfurnished       73491
Furnished         19421
NaN                2067
Name: count, dtype: int64

 Transaction
Transaction
Resale          135630
New Property     41445
Other              703
NaN                 67
Rent/Lease           2
Name: count, dtype: int64

 Status
Status
Ready to Move    177252
NaN                 595
Name: count, dtype: int64


In [10]:
# Mode fills (safe; all have valid modes)
df['Furnishing'] = df['Furnishing'].fillna(df['Furnishing'].mode()[0])
df['Transaction'] = df['Transaction'].fillna(df['Transaction'].mode()[0])
df['Status'] = df['Status'].fillna(df['Status'].mode()[0])

In [11]:
df = df.drop(columns=['Amount(in rupees)'])

In [12]:
print(df.shape)
print(df.isnull().sum())
print(df.dtypes)

(177847, 21)
Index                     0
Title                     0
Description            2929
Price (in rupees)      7981
location                  0
Carpet Area           76325
Status                    0
Floor                  6949
Transaction               0
Furnishing                0
facing                65736
overlooking           75825
Society              102919
Bathroom                760
Balcony               47159
Car Parking           95912
Ownership             61925
Super Area           101612
Dimensions           177847
Plot Area            177847
price                     0
dtype: int64
Index                  int64
Title                    str
Description              str
Price (in rupees)    float64
location                 str
Carpet Area              str
Status                   str
Floor                    str
Transaction              str
Furnishing               str
facing                   str
overlooking              str
Society                  str
Bathroom 

In [13]:
df = df.drop(columns=[
    'Index','Title','Description',
    'Price (in rupees)','facing','overlooking',
    'Society','Ownership','Super Area',
    'Dimensions','Plot Area'
])

In [14]:
df['Carpet Area'] = df['Carpet Area'].astype(str).str.replace('sqft', '', regex=False)
df['Carpet Area'] = pd.to_numeric(df['Carpet Area'], errors='coerce')
df['Carpet Area'] = df['Carpet Area'].fillna(df['Carpet Area'].median())

In [15]:
df['Bathroom'] = pd.to_numeric(df['Bathroom'], errors='coerce')
df['Bathroom'] = df['Bathroom'].fillna(df['Bathroom'].median())
df['Bathroom'] = df['Bathroom'].clip(upper=6)

In [16]:
df['Balcony'] = df['Balcony'].astype(str).str.replace('> ', '', regex=False)
df['Balcony'] = pd.to_numeric(df['Balcony'], errors='coerce')
df['Balcony'] = df['Balcony'].fillna(0)

In [17]:
df['Floor'] = df['Floor'].astype(str).str.extract(r'(\d+)')
df['Floor'] = pd.to_numeric(df['Floor'], errors='coerce')
df['Floor'] = df['Floor'].fillna(df['Floor'].median())
df['Floor'] = df['Floor'].clip(upper=30)

In [18]:
df['Car Parking'] = df['Car Parking'].notnull().astype(int)

In [19]:
df['Furnishing'] = df['Furnishing'].fillna(df['Furnishing'].mode()[0])
df['Transaction'] = df['Transaction'].fillna(df['Transaction'].mode()[0])
df['Status'] = df['Status'].fillna(df['Status'].mode()[0])

In [20]:
print(df.shape)
print(df.isnull().sum())
print(df.dtypes)

(177847, 10)
location       0
Carpet Area    0
Status         0
Floor          0
Transaction    0
Furnishing     0
Bathroom       0
Balcony        0
Car Parking    0
price          0
dtype: int64
location           str
Carpet Area    float64
Status             str
Floor          float64
Transaction        str
Furnishing         str
Bathroom       float64
Balcony        float64
Car Parking      int64
price          float64
dtype: object


In [21]:
df = df[(df['Carpet Area'] >= 300) & (df['Carpet Area'] <= 3000)]

print(df.shape)

(173543, 10)


In [22]:
df['Carpet Area'].describe()

count    173543.000000
mean       1148.990141
std         369.599598
min         300.000000
25%        1050.000000
50%        1100.000000
75%        1130.000000
max        3000.000000
Name: Carpet Area, dtype: float64

In [23]:
print(df.shape)
print(df.isnull().sum())
print(df.dtypes)

(173543, 10)
location       0
Carpet Area    0
Status         0
Floor          0
Transaction    0
Furnishing     0
Bathroom       0
Balcony        0
Car Parking    0
price          0
dtype: int64
location           str
Carpet Area    float64
Status             str
Floor          float64
Transaction        str
Furnishing         str
Bathroom       float64
Balcony        float64
Car Parking      int64
price          float64
dtype: object


In [24]:
num_cols = ['Carpet Area','Floor','Bathroom','Balcony','Car Parking','price']
df[num_cols].describe()

,Carpet Area,Floor,Bathroom,Balcony,Car Parking,price
count,173543.000000,173543.000000,173543.000000,173543.000000,173543.000000,173543.000000
mean,1148.990141,4.744611,2.420737,1.472500,0.454861,15.904338
std,369.599598,4.424895,0.801901,1.203026,0.497960,0.790309
min,300.000000,1.000000,1.000000,0.000000,0.000000,11.512925
25%,1050.000000,2.000000,2.000000,0.000000,0.000000,15.388284
50%,1100.000000,3.000000,2.000000,1.000000,0.000000,15.844974
75%,1130.000000,6.000000,3.000000,2.000000,1.000000,16.454568
max,3000.000000,30.000000,6.000000,10.000000,1.000000,23.362537


In [25]:
df['Balcony'] = df['Balcony'].clip(upper=5)

In [26]:
df = df[df['price'] <= 18]

In [27]:
X = df.drop('price', axis=1)
y = df['price']

In [28]:
df['Bathroom'] = df['Bathroom'].astype(int)

In [29]:
df['Balcony'] = df['Balcony'].astype(int)

In [30]:
df['Floor'] = df['Floor'].astype(int)
df['Car Parking'] = df['Car Parking'].astype(int)

In [31]:


from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [32]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_cols = ['Carpet Area', 'Bathroom', 'Balcony', 'Floor', 'Car Parking']
cat_cols = ['location', 'Furnishing', 'Transaction', 'Status']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

In [33]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

lr = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

lr.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](9,)","['location','Carpet Area','Status',...,'Bathroom','Balcony','Car Parking']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,9
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of column

In [34]:
from sklearn.ensemble import RandomForestRegressor

rf = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42))
])

rf.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](9,)","['location','Carpet Area','Status',...,'Bathroom','Balcony','Car Parking']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,9
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of column

In [36]:
from xgboost import XGBRegressor

xgb = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1))
])

xgb.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](9,)","['location','Carpet Area','Status',...,'Bathroom','Balcony','Car Parking']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,9
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of column

In [37]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

def evaluate(model, name):
    y_pred = model.predict(X_test)
    print(name)
    print("R2:", r2_score(y_test, y_pred))
    print("MAE:", mean_absolute_error(y_test, y_pred))
    print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
    print("-"*30)

evaluate(lr, "Linear Regression")
evaluate(rf, "Random Forest")
evaluate(xgb, "XGBoost")

Linear Regression
R2: 0.6671462031559448
MAE: 0.3360475309023266
RMSE: 0.44239322705638867
------------------------------
Random Forest
R2: 0.8854718582742739
MAE: 0.12316407550423958
RMSE: 0.2595002728489953
------------------------------
XGBoost
R2: 0.8230782023888987
MAE: 0.228772497623152
RMSE: 0.3225316834882712
------------------------------


In [38]:
best_model = rf

In [39]:
import joblib
joblib.dump(best_model, "model.pkl")

['model.pkl']

In [40]:
df.columns

Index(['location', 'Carpet Area', 'Status', 'Floor', 'Transaction',
       'Furnishing', 'Bathroom', 'Balcony', 'Car Parking', 'price'],
      dtype='str')

In [41]:
df['Carpet Area'].max()

np.float64(3000.0)

In [42]:
df['Floor'].unique()

array([10,  3,  1, 20,  2,  4,  7,  6, 16,  8, 18,  5, 15, 27, 11,  9, 14,
       12, 21, 29, 13, 19, 17, 28, 30, 23, 25, 26, 24, 22])

In [43]:
df['Car Parking'].unique()

array([0, 1])

In [44]:
(df['Floor'] % 1).sum()

np.int64(0)